In [ ]:
%load_ext autoreload
%autoreload 2
%cd ..

In [ ]:
import pandas as pd
from mol_gen_docking.data.pydantic_dataset import read_jsonl, write_jsonl, Sample, Message, Conversation
from pathlib import Path
import jsonlines
from typing import Dict, Any, List, Tuple, Iterator
import itertools
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from tqdm.auto import tqdm

from mol_gen_docking.evaluation.sft_extraction import SFTExtractionConfig, SFTExtractor, Completion
# Remove rdkit logging
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

In [ ]:
DATASET = "molgendata_train"
PROMPT_PATH = Path(f"data/{DATASET.replace('_train', '')}/train_data/train_prompts_boxed.jsonl")

prompt_dataset = read_jsonl(PROMPT_PATH)
SOURCES = ["Qwen3-30B-A3B-Thinking-2507", "MiniMax-M2"]
comp_dataset = {
    source: [] for source in SOURCES
}
comp_dataset["both"] = []

for source in SOURCES:
    PATH = Path("MolGenOutput") / DATASET / source
    files = sorted(list(PATH.glob("*_scored.jsonl")))
    n_files = len(files)
    i = 0
    for path in tqdm(files, total=n_files):
        with jsonlines.open(path) as reader:
            for obj in reader:
                obj["source"] = source
                obj["metadata"]["prompt_id"] = prompt_dataset[i//16].identifier
                obj  = Completion.model_validate(obj)
                comp_dataset["both"].append(obj)
                comp_dataset[source].append(obj)
                i+=1


    print(f"Extracted {len(comp_dataset['both'])} completions from {n_files} files.")

In [ ]:
def grid_translate(grid: Dict[str, List[Any]]) -> Iterator[SFTExtractionConfig]:
    keys, values = zip(*grid.items())
    for v in itertools.product(*values):
        config_dict = {}
        for key, value in zip(keys, v):
            if not value == "default":
                config_dict[key] = value
        yield SFTExtractionConfig(**config_dict)

grid0 = {
    "min_reward_threshold": [None, 0.3],
    "div_threshold": [None, 0.4],
    "reward_info_template": [{}],
    "source_info_template": [{},], #"default"],
    "system_prompt_path": ["system_prompts/vanilla_boxed.json"],
}

In [ ]:
def create_logging_df(config: SFTExtractionConfig, extractor: SFTExtractor, source: str):
    # Get metadatas
    df = pd.DataFrame(extractor.metadata.model_dump())
    # Create bins for n_tokens
    bins_tokens = [0,1000, 2000, 5000, 7000, 10000, 1e8]
    bins_lab = ["<1k", "1k-2k", "2k-5k", "5k-7k", "7k-10k", ">10k"]
    bins = pd.cut(df["n_tokens"], bins=bins_tokens, right=False, labels=bins_lab)
    df["n_tokens_bins"] =bins
    df["min_reward_threshold"] = config.min_reward_threshold if config.min_reward_threshold is not None else 0.
    df["div_threshold"] = config.div_threshold if config.div_threshold is not None else 1.
    if source == "both":
        df["source_temp"] = "empty" if config.source_info_template == {} else "default"
    else:
        df["source_temp"] = source
    return df

def config_to_dir(config: SFTExtractionConfig, source: str) -> Path:
    trace_path = Path("data/traces")
    for key, value in config.model_dump().items():
        if not key in ["system_prompt_path", "reward_info_template"]: # Kept cst
            if key == "source_info_template":
                if source != "both":
                    assert value == {}
                    trace_path /= source
                    continue
                if isinstance(value, dict):
                    if len(value) == 0:
                        value = "empty"
                    else:
                        value = f"dict_{len(value)}"
                elif isinstance(value, str):
                    value = value.split("/")[-1].replace(".json", "")
            trace_path /= f"{key}_{value}"
    return trace_path

In [ ]:
dfs = []
for source in ["Qwen3-30B-A3B-Thinking-2507", "MiniMax-M2", "both"]:
    for config in grid_translate(grid0):
        extractor = SFTExtractor(config)
        new_dataset = extractor.extract(comp_dataset[source], prompt_dataset)
        # Save traces
        trace_dir = config_to_dir(config, source)
        trace_path = trace_dir / f"{DATASET}.jsonl"
        trace_path.parent.mkdir(parents=True, exist_ok=True)
        write_jsonl(trace_path, new_dataset)
        print(f"N samples extracted for config:\n === {config.model_dump()} ===\n**{sum([len(sample.conversations) for sample in new_dataset])}**")

        dfs.append(create_logging_df(config, extractor, source))

full_df = pd.concat(dfs).reset_index(drop=True)

In [ ]:
full_df

In [ ]:
# Grouped size and number of unique prompt_ids
grouped = full_df.groupby(["min_reward_threshold", "div_threshold", "source_temp"]).agg(
    size=("prompt_ids", "size"),
    unique_prompt_ids=("prompt_ids", "nunique")
).reset_index()
# Melt the dataframe for plotting
melted = grouped.melt(id_vars=["min_reward_threshold", "div_threshold", "source_temp"], value_vars=["size", "unique_prompt_ids"], var_name="metric", value_name="value")

facet_grid = sns.FacetGrid(melted, hue="min_reward_threshold", row="metric", col="source_temp", margin_titles=True, sharex=True, sharey="row", height=3, aspect=2)
facet_grid.map_dataframe(
    sns.barplot,
    x="div_threshold",
    y="value",
    # multiple="dodge",
)
facet_grid.set_titles(
    col_template="$r_m$={col_name}",
    row_template="{row_name}"
)
# Add legend outside of the plot
facet_grid.add_legend(title="$r_m$", bbox_to_anchor=(0.9, 0.5), loc="center left")


In [ ]:
# Plot the distribution of the reward
facet = sns.FacetGrid(full_df[full_df.source_temp != "empty"], col="min_reward_threshold", row="div_threshold", hue="source_temp", margin_titles=True, sharex=True, sharey=True, height=1.5, aspect=1.8, legend_out=True)
facet.map_dataframe(
    sns.histplot, x="rewards", bins=20, multiple="stack", stat="probability", binwidth=0.1, alpha = 0.7
)
facet.set_titles(col_template="$r_m$={col_name}", row_template="$t_d$={row_name}")
# Add legend outside of the plot
facet.add_legend(title="Source", bbox_to_anchor=(0.7, 0.5), loc="center left")


In [ ]:
# Plot distribution of n_tokens
facet = sns.FacetGrid(full_df[full_df.source_temp == "MiniMax-M2"], col="min_reward_threshold", row="div_threshold", margin_titles=True, sharex=True, sharey=True, height=1.5, aspect=1.8)
facet.map_dataframe(
    sns.histplot, x="n_tokens", bins=20, hue="n_tokens_bins", multiple="stack", palette="flare_r", stat="probability"
)
facet.set_titles(col_template="$r_m$={col_name}", row_template="$t_d$={row_name}")
# max x: 20000
facet.set(xlim=(0, 20000))
# Global title
facet.fig.suptitle("MiniMax-M2")

In [ ]:
# Plot distribution of n_tokens
facet = sns.FacetGrid(full_df[full_df.source_temp == "Qwen3-30B-A3B-Thinking-2507"], col="min_reward_threshold", row="div_threshold", margin_titles=True, sharex=True, sharey=True, height=1.5, aspect=1.8)
facet.map_dataframe(
    sns.histplot, x="n_tokens", bins=20, hue="n_tokens_bins", multiple="stack", palette="flare_r", stat="probability"
)
facet.set_titles(col_template="$r_m$={col_name}", row_template="$t_d$={row_name}")
# max x: 20000
# facet.set(xlim=(0, 20000))
# Global title
facet.fig.suptitle("Qwen3-30B-A3B-Thinking-2507")